# 50 - Evaluation of the full run: ablation matrix, shuffle check, tables (S4.3, S3.4 gates, S4.6)

Thin driver over `citecurv` for the plan `docs/superpowers/plans/2026-11-25-s43-ablation-matrix.md`. Every step
(C0 to C6) starts on Maor's word after the previous step's report; run the cells of ONE step per session.

- Runtime: Colab CPU High-RAM, runtime version 2026.07 (Python 3.12, the GraphRicciCurvature pins). Step C5b alone
  runs on a GPU runtime of the image that made seed 0's stored TGN scores (it needs no ORC library).
- Inputs on Drive `MyDrive/citecurv/full/`: `events.parquet`, `works.parquet`, `event_labels.parquet` (with
  `via_src`, `via_dst`), `embeddings.npz`, `embeddings_cited.npz`, `cited_joint.parquet`, `split.yaml`, `tgn/data/` (DyGLib
  files), `tgn/runs/TGN_full_seed<s>/`, `tgn/infer/TGN_full_seed<s>/`.
- Heavy ORC work runs in a child process (`python -m citecurv.eval.orc_job`) that this kernel watches through a
  heartbeat on local disk; ORC cells run BEFORE the inputs cell of their session.
- Every job cell has a `MAX_HOURS` watchdog and ends with `drive.flush_and_unmount()` and a runtime release.

In [ ]:
# 1. Runtime guard and BLAS pin, before numpy is imported (same kernels on every host: NFR4 across sessions)
import os
import platform
import sys

os.environ["OPENBLAS_CORETYPE"] = "Haswell"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
print("python", platform.python_version())
if sys.version_info[:2] != (3, 12) and not os.environ.get("CITECURV_GPU_STEP"):
    raise RuntimeError("CPU steps need Python 3.12 (runtime version 2026.07); set CITECURV_GPU_STEP=1 only for C5b")

In [ ]:
# 2. Install the repository at one commit (editable, so configs/ and third_party/ sit next to the package)
COMMIT = "SET-COMMIT"  # SET: the pushed commit this run uses (recorded in every run's TAGS)
REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
REPO_DIR = "/content/citecurv"
assert COMMIT != "SET-COMMIT", "set COMMIT first"
!rm -rf {REPO_DIR} && git clone -q {REPO} {REPO_DIR} && git -C {REPO_DIR} checkout -q {COMMIT}
!pip install -q -e {REPO_DIR} pyyaml scikit-learn xgboost
if not os.environ.get("CITECURV_GPU_STEP"):
    !pip install -q GraphRicciCurvature==0.6.1 "scipy<=1.13.1" POT==0.9.6 networkit==11.2.1 networkx
    from importlib.metadata import version
    want = {"GraphRicciCurvature": "0.6.1", "POT": "0.9.6", "networkit": "11.2.1"}
    got = {k: version(k) for k in want}
    assert got == want, got
    assert version("scipy") <= "1.13.1", version("scipy")
# This kernel started before `pip install -e` wrote its .pth file, so it would find the repo root (REPO_DIR) as a
# namespace package `citecurv` without `citecurv.eval` (Colab 2026-11-25, C3). Put src first and drop any stale entry.
import importlib
sys.path.insert(0, f"{REPO_DIR}/src")
for _m in [m for m in sys.modules if m == "citecurv" or m.startswith("citecurv.")]:
    del sys.modules[_m]
importlib.invalidate_caches()
os.environ["CITECURV_DYGLIB_ROOT"] = f"{REPO_DIR}/third_party/DyGLib"

In [ ]:
# 3. Helpers: memory log, host fingerprint, watchdog and release, the ORC child job
import json
import signal
import subprocess
import threading
import time
from pathlib import Path


class MemLog:
    """Peak resident memory per phase, from /proc/self/status read every second (works without clear_refs)."""

    def __init__(self):
        self.phase, self.peaks, self._stop = "start", {}, False
        threading.Thread(target=self._run, daemon=True).start()

    @staticmethod
    def rss_gb():
        for line in open("/proc/self/status"):
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) / 1e6
        return float("nan")

    def _run(self):
        while not self._stop:
            r = self.rss_gb()
            self.peaks[self.phase] = max(self.peaks.get(self.phase, 0.0), r)
            time.sleep(1)

    def set(self, phase):
        self.phase = phase
        try:
            open("/proc/self/clear_refs", "w").write("5")  # optional; not every sandbox has it
        except OSError:
            pass


def host_fingerprint():
    import numpy as np

    cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "?")
    import contextlib
    import io

    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            np.show_runtime()  # prints SIMD extensions and the BLAS core; returns None
    except Exception as e:
        buf.write(repr(e))
    return {"cpu": cpu, "vcpus": os.cpu_count(), "numpy": np.__version__, "numpy_runtime": buf.getvalue()[:3000]}


def start_watchdog(max_hours, why="time limit"):
    def fire():
        print("MAX_HOURS reached:", why, flush=True)
        release(why)
    t = threading.Timer(max_hours * 3600, fire)
    t.daemon = True
    t.start()
    return t


def release(why):
    from google.colab import drive, runtime

    print("release:", why, flush=True)
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print("flush failed", e)
    runtime.unassign()


def run_orc_job(job, *, setup_limit_s, batch_limit_s, poll_s=30, restarts=1):
    """Start citecurv.eval.orc_job in its own process group and kill the whole group (worker and pool included)
    when its heartbeat is older than its phase's limit. A killed job is restarted at most `restarts` times; the
    restarted job marks the rows in flight at the kill as errors. Returns the job's result JSON."""
    jp = Path("/content") / f"job_{int(time.time())}.json"
    jp.write_text(json.dumps(job))
    hb = Path(job["heartbeat"])
    for attempt in range(restarts + 1):
        proc = subprocess.Popen([sys.executable, "-m", "citecurv.eval.orc_job", str(jp)], start_new_session=True)
        while proc.poll() is None:
            time.sleep(poll_s)
            try:
                beat = json.loads(hb.read_text())
                age, phase = time.time() - beat.get("time", time.time()), beat.get("phase")
            except (OSError, ValueError):
                age, phase = 0, "load"
            limit = batch_limit_s if phase == "orc" else setup_limit_s
            if age > limit:
                print(f"heartbeat {age:.0f} s old in phase {phase}: killing the job group", flush=True)
                os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                proc.wait()
                break
        if proc.returncode == 0:
            return json.loads(Path(job["result"]).read_text())
        print("job ended with", proc.returncode, "attempt", attempt, flush=True)
    raise RuntimeError("ORC job failed twice: report to Maor before anything else")


MEM = MemLog()

In [ ]:
# 4. Drive, paths, run tags
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/citecurv/full")
EVAL = DRIVE / "eval"
EVAL.mkdir(parents=True, exist_ok=True)
CONFIG = Path(REPO_DIR) / "configs" / "default.yaml"
SPLIT_YAML = DRIVE / "split.yaml"  # where S4.1 wrote it (checked on Drive 2026-11-25)
SEEDS = [0, 1, 2, 3, 4]
CELLS = ["full", "no_curvature", "forman_only", "no_temporal", "full_learned"]  # ollivier_only comes in C4
import yaml

CUTOFF = int(yaml.safe_load(SPLIT_YAML.read_text())["cutoff_t"])
LOCAL = Path("/content/local")
LOCAL.mkdir(exist_ok=True)
# option A (Maor 2026-11-26): the closed-form directed ORC matrix writes here. The C3 run of 2026-11-25 stays
# under EVAL/matrix/real and EVAL/matrix/no_temporal_keep_temp as the plan-formulation record; no other cell
# names those folders (tests/test_eval_notebook.py).
MATRIX_ROOT = EVAL / "matrix" / "real_orcdir"
KEEP_ROOT = EVAL / "matrix" / "no_temporal_keep_temp_orcdir"


def base_job(**kw):
    return {"events": str(DRIVE / "events.parquet"), "works": str(DRIVE / "works.parquet"),
            "cited_joint": str(DRIVE / "cited_joint.parquet"), "shuffle": None, "config": str(CONFIG),
            "cutoff": CUTOFF, "seeds": [], "out_path": str(EVAL / "orc_matrix.parquet"), "known": None,
            "known_max_t": None, "batch": 1, "chunk": 5000, "worker": False, "order_seed": 0, "max_chunks": None,
            "heartbeat": str(LOCAL / "heartbeat.json"), "result": str(LOCAL / "orc_result.json"), **kw}


def tags_for(seed, extra=None, memo_path=None):
    """Run tags: commit, TGN folder and best epoch, host fingerprint, the ORC memo file's sha256 and the context
    rows' hash (Global Constraints of the plan)."""
    import hashlib

    from citecurv.pipeline import context_hash

    meta = json.loads((DRIVE / "tgn" / "runs" / f"TGN_full_seed{seed}" / "metrics.json").read_text())
    memo_path = Path(memo_path or EVAL / "orc_matrix.parquet")
    sha = None
    if memo_path.exists():
        h = hashlib.sha256()
        with open(memo_path, "rb") as f:
            for block in iter(lambda: f.read(1 << 22), b""):
                h.update(block)
        sha = h.hexdigest()
    ctx = INPUTS.get("context") if "INPUTS" in globals() else None
    return {"group": "matrix", "level": "real", "commit": COMMIT, "tgn_infer": f"tgn/infer/TGN_full_seed{seed}",
            "best_epoch": meta.get("best_epoch"), "host": host_fingerprint(), "orc_memo_sha256": sha,
            "context_hash": context_hash(ctx) if ctx is not None else None, **(extra or {})}

In [ ]:
# 5. Inputs for matrix cells (run AFTER any ORC job cell of this session, or after its frames are freed)
import gc

import numpy as np
import pandas as pd

from citecurv.config import load_config
from citecurv.eval.split import split_counts, split_events
from citecurv.pipeline import (_canonical_events, build_context, context_hash, data_hash, load_orc_memo,
                               orc_settings_hash, resolve_config)
from citecurv.tgn.infer import load_event_scores

INPUTS = {}


def load_inputs():
    MEM.set("inputs")
    print("VmRSS before inputs", round(MemLog.rss_gb(), 2), "GB")
    events = pd.read_parquet(DRIVE / "events.parquet", columns=["event_id", "src_id", "dst_id", "t"])
    works = pd.read_parquet(DRIVE / "works.parquet")
    labels = pd.read_parquet(DRIVE / "event_labels.parquet")
    with np.load(DRIVE / "embeddings.npz", allow_pickle=False) as a, \
            np.load(DRIVE / "embeddings_cited.npz", allow_pickle=False) as b:
        ids = np.concatenate([a["ids"].astype(str), b["ids"].astype(str)])
        vectors = np.concatenate([a["vectors"], b["vectors"]])
    refs = pd.read_parquet(DRIVE / "cited_joint.parquet", columns=["work_id", "pub_date", "ref_ids"])
    context, ctx_stats = build_context(works, refs, events)
    del refs
    gc.collect()
    cfg = load_config(CONFIG)
    rcfg = resolve_config(cfg)
    ev = _canonical_events(events)
    split = split_events(ev, CUTOFF, val_frac=rcfg["split"]["val_frac"])
    want = yaml.safe_load(SPLIT_YAML.read_text())["counts"]
    got = split_counts(ev, split)
    for k in ("n_train", "n_val", "n_test"):
        if int(got[k]) != int(want[k]):
            raise RuntimeError(f"split check failed: {k} {got[k]} here, {want[k]} in split.yaml")
    INPUTS.update(events=events, works=works, labels=labels, embeddings=(ids, vectors), context=context, cfg=cfg,
                  rcfg=rcfg, data=dict(events=events, works=works, embeddings=(ids, vectors), event_labels=labels,
                                       cutoff=CUTOFF))
    print("inputs:", len(events), "events,", len(context), "context rows; split ok; VmRSS", round(MemLog.rss_gb(), 2))
    return INPUTS


def coupling():
    """reference_overlap of the run's events with the context rows, computed once per session (option A): every
    closed cell gets it through run_matrix(..., coupling=coupling()), before the TGN arrays of a seed load."""
    if "coupling" not in INPUTS:
        from citecurv.curvature.overlap import reference_overlap

        MEM.set("coupling")
        INPUTS["coupling"] = reference_overlap(_canonical_events(INPUTS["events"]), context=INPUTS["context"])
        print("coupling:", len(INPUTS["coupling"]), "rows; VmRSS", round(MemLog.rss_gb(), 2), "GB", flush=True)
    return INPUTS["coupling"]


def free_inputs():
    INPUTS.clear()
    gc.collect()


def tgn_inputs(seed):
    d = DRIVE / "tgn" / "infer" / f"TGN_full_seed{seed}"
    with np.load(d / "tgn_event_embeddings.npz") as z:
        emb = {k: z[k] for k in z.files}
    return {"tgn_scores": load_event_scores(d), "tgn_embeddings": emb}


def memo(path):
    ev = _canonical_events(INPUTS["events"])
    return load_orc_memo(path, data_hash=data_hash(ev), context_hash=context_hash(INPUTS["context"]),
                         settings_hash=orc_settings_hash(INPUTS["rcfg"]), radius=INPUTS["rcfg"]["curvature"]["orc_radius"])

## C0 - the suite on this runtime

In [ ]:
!cd {REPO_DIR} && python -m pytest -q -rA 2>&1 | tail -40

## C1 - measurements in the precompute's memory state (no vectors, no TGN in the job)

Short ORC jobs (one chunk each) in a seeded random order over the targets, so their rows are an unbiased draw
weighted by the real counts. Report every number to Maor before C2.

In [ ]:
MAX_HOURS = 3
watchdog = start_watchdog(MAX_HOURS)
probes = {}
for batch, worker in [(1, False), (10, False), (50, False), (1, True), (10, True)]:
    name = f"b{batch}_{'w' if worker else 'in'}"
    job = base_job(seeds=[0], batch=batch, worker=worker, chunk=500, max_chunks=1,
                   out_path=str(EVAL / "c1" / f"orc_probe_{name}.parquet"),
                   heartbeat=str(LOCAL / f"hb_{name}.json"), result=str(LOCAL / f"res_{name}.json"))
    Path(job["out_path"]).parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    probes[name] = {**run_orc_job(job, setup_limit_s=4 * 3600, batch_limit_s=1800), "wall_s": time.time() - t0}
    print(name, {k: probes[name].get(k) for k in ("n_targets", "n_done", "subgraph_s_per_event", "orc_s_per_event",
                                                    "n_errors", "wall_s")})
first = probes["b1_in"]
print("split", first["split_counts"], "| screen flagged", first["n_screen_flagged"], "by split",
      first.get("screen_flagged_by_split"), "| G13 unknown", first.get("n_screen_unknown"), "| eligible",
      first["n_eligible"])
for name, pr in probes.items():
    print(name, "subgraph edges p95 / max:", pr.get("p95_subgraph_edges"), "/", pr.get("max_subgraph_edges"))
print("C6 projection: timed on its first comparison (no run folders exist before C3)")
# cold start, D4 cell and screen-escalated test positives, from the events alone (cheap)
import pandas as pd

from citecurv.config import load_config
from citecurv.pipeline import _canonical_events, resolve_config

events = pd.read_parquet(DRIVE / "events.parquet", columns=["event_id", "src_id", "dst_id", "t"])
labels = pd.read_parquet(DRIVE / "event_labels.parquet")
ev = _canonical_events(events)
k = resolve_config(load_config(CONFIG))["scoring"]["cold_start_k"]
prior = ev.groupby("dst_id").cumcount() - ev.groupby(["dst_id", "t"]).cumcount()  # citations of dst before t
cold = prior.to_numpy() < k
test = ev["t"].to_numpy() > CUTOFF
y = ev["event_id"].map(labels.set_index("event_id")["label"]).fillna(0).to_numpy(dtype=bool)
print("test positives", int((y & test).sum()), "| of them cold-start", int((y & test & cold).sum()))
lab = ev["event_id"].map(labels.set_index("event_id")["via_src"]).fillna(0).to_numpy(dtype=bool), \
    ev["event_id"].map(labels.set_index("event_id")["via_dst"]).fillna(0).to_numpy(dtype=bool)
print("cold-start test positives via citing / cited paper:", int((lab[0] & test & cold).sum()),
      int((lab[1] & test & cold).sum()))
works = pd.read_parquet(DRIVE / "works.parquet")
if "field" in works.columns:
    fld = works.set_index("work_id")["field"]
    field_name = "field"
else:
    fld = works.set_index("work_id")["concepts"].map(lambda c: (list(c) or ["?"])[0])
    field_name = "first concept"
cells_ = ev.loc[test & ~cold, "src_id"].map(fld).value_counts(normalize=True)
print("D4 uses", field_name, "| largest cell share of scored test rows:", round(float(cells_.iloc[0]), 3),
      cells_.index[0])
(EVAL / "c1" / "probes.json").write_text(json.dumps(probes, indent=1, default=str))
watchdog.cancel()

## C2 - ORC precompute, screen set (child job, resumable, may run overnight)

In [ ]:
BATCH, WORKER = 10, True  # SET from C1
MAX_HOURS = 20
watchdog = start_watchdog(MAX_HOURS)
res = run_orc_job(base_job(seeds=[], batch=BATCH, worker=WORKER), setup_limit_s=4 * 3600, batch_limit_s=1800)
print({k: res.get(k) for k in ("n_targets", "n_done", "n_pending", "n_errors", "n_attempted_nan", "seconds")})
(EVAL / "c2_result.json").write_text(json.dumps(res, indent=1, default=str))
release("C2 finished")

## C3 - matrix without ollivier_only (one run_matrix call per seed); full_0 gates in this session

In [ ]:
from citecurv.eval.ablation import run_matrix, write_run
from citecurv.pipeline import gate_cold_start, gate_fairness, gate_top_breakdown

MAX_HOURS = 20
watchdog = start_watchdog(MAX_HOURS)
load_inputs()
GATES = {}


def keep_gates(res):
    MEM.set(f"{res.variant}_{res.seed}")
    if res.variant == "full" and res.seed == 0:
        GATES.update(cold_start=gate_cold_start(res), top20=gate_top_breakdown(res, k=20),
                     d4=gate_fairness(res, INPUTS["works"]))
        write_run(MATRIX_ROOT / "runs" / "gates_full_0",
                  {"kind": "sanity", "method": "gates", "seed": 0, "tags": {"group": "full_run_gates", "level": "real"},
                   "test": None, "gates": GATES})
    print(res.variant, res.seed, res.metrics["test"]["ap"], res.weights_source, MEM.peaks, flush=True)


# integration smoke: full seed 0 alone first (the closed-form path); its peak, the coupling included, is the
# memory baseline of every later cell
run_matrix(["full"], [0], lambda s: {**INPUTS["data"], **tgn_inputs(s)}, MATRIX_ROOT,
           cfg=INPUTS["cfg"], tags=tags_for(0), resume=True, identity_data=INPUTS["data"],
           context=INPUTS["context"], coupling=coupling(), callback=keep_gates)
SMOKE_PEAK = max(MEM.peaks.values())
print("smoke peak", round(SMOKE_PEAK, 2), "GB")
for seed in SEEDS:
    if MemLog.rss_gb() > SMOKE_PEAK + 2:
        raise RuntimeError(f"VmRSS {MemLog.rss_gb():.1f} GB above the smoke baseline + 2 GB: stop, report to Maor")
    run_matrix(CELLS, [seed], lambda s: {**INPUTS["data"], **tgn_inputs(s)}, MATRIX_ROOT,
               cfg=INPUTS["cfg"], tags=tags_for(seed), resume=True, identity_data=INPUTS["data"],
               context=INPUTS["context"], coupling=coupling(), callback=keep_gates)
    keep_cfg = json.loads(json.dumps(INPUTS["cfg"]))
    keep_cfg.setdefault("pipeline", {})["no_temporal_drop_s_temp"] = False
    run_matrix(["no_temporal"], [seed], lambda s: {**INPUTS["data"], **tgn_inputs(s)},
               KEEP_ROOT, cfg=keep_cfg, tags=tags_for(seed, {"group": "no_temporal_keep_temp"}), resume=True,
               identity_data=INPUTS["data"], context=INPUTS["context"], coupling=coupling())
print("peaks", MEM.peaks)
release("C3 finished")

In [ ]:
# C3 reproducibility (fresh session): full seed 0 again (the closed-form path), into its own folder (group repro)
from citecurv.eval.ablation import run_matrix, write_run
from citecurv.pipeline import gate_reproducibility

MAX_HOURS = 6
watchdog = start_watchdog(MAX_HOURS)
load_inputs()
run_matrix(["full"], [0], lambda s: {**INPUTS["data"], **tgn_inputs(s)}, EVAL / "repro_orcdir", cfg=INPUTS["cfg"],
           tags=tags_for(0, {"group": "repro"}), context=INPUTS["context"], coupling=coupling())
g = gate_reproducibility(MATRIX_ROOT / "runs" / "full_0" / "scores.parquet",
                         EVAL / "repro_orcdir" / "runs" / "full_0" / "scores.parquet")
a = json.loads((MATRIX_ROOT / "runs" / "full_0" / "metrics.json").read_text())["tags"]["host"]
now = host_fingerprint()
write_run(MATRIX_ROOT / "runs" / "gates_repro_0",
          {"kind": "sanity", "method": "gates", "seed": 0, "tags": {"group": "full_run_gates", "level": "real"},
           "test": None, "gates": {"reproducibility": g}, "host_then": a, "host_now": now,
           "same_host": a.get("cpu") == now.get("cpu") and a.get("numpy_runtime") == now.get("numpy_runtime")})
print(g["status"], "| same host:", a.get("cpu") == now.get("cpu"), "| on different hosts rerun on a matching host")
release("C3 repro finished")

## C4 - random sets and ollivier_only, one seed at a time (job first, then the cell)

In [ ]:
raise RuntimeError("C4 dropped (option A, Maor 2026-11-26): the matrix cells compute the directed ORC in closed "
                   "form; no ORC job and no ollivier_only cell run. The code below is the plan's record.")
from citecurv.eval.ablation import run_matrix

BATCH, WORKER = 10, True  # SET from C1
MAX_HOURS = 20
watchdog = start_watchdog(MAX_HOURS)
for seed in SEEDS:
    free_inputs()
    print("VmRSS before the job", round(MemLog.rss_gb(), 2), "GB")
    res = run_orc_job(base_job(seeds=list(range(seed + 1)), batch=BATCH, worker=WORKER),
                      setup_limit_s=4 * 3600, batch_limit_s=1800)
    print("seed", seed, {k: res.get(k) for k in ("n_targets", "n_done", "n_pending", "n_errors")})
    load_inputs()
    MEMO = memo(EVAL / "orc_matrix.parquet")
    run_matrix(["ollivier_only"], [seed], lambda s: {**INPUTS["data"], **tgn_inputs(s)}, MATRIX_ROOT,
               cfg=INPUTS["cfg"], orc_memo=MEMO, tags=tags_for(seed), resume=True, require_orc_memo=True,
               identity_data=INPUTS["data"], context=INPUTS["context"])
    del MEMO
release("C4 finished")

## C5 - timestamp-shuffle check with the TGN re-scored on the shuffled stream

C5a (CPU): shuffled DyGLib files and the shuffled ORC memo. C5b (GPU, the image of seed 0's stored scores): TGN
scoring of the shuffled stream. C5c (CPU, one session): base `full` seed 0 on the original stream, merge, shuffled
`full` run, gate. Expected readings are in docs/colab_runbook.md (written before this step runs). A FAIL or a drift
above the tolerance stops the plan: report to Maor, no tables until he rules.

In [ ]:
# C5a (CPU)
from citecurv.pipeline import shuffle_test_timestamps
from citecurv.tgn.convert import shuffle_dyglib_dataset

SHUF_SEED = 777
events = pd.read_parquet(DRIVE / "events.parquet", columns=["event_id", "src_id", "dst_id", "t"])
sh = shuffle_test_timestamps(events, CUTOFF, seed=SHUF_SEED)
meta = shuffle_dyglib_dataset(DRIVE / "tgn" / "data", "full", sh[["event_id", "t"]], DRIVE / "tgn" / "data_shuf777",
                              "full_shuf777")
print(meta["n_moved"], "events moved")
del events, sh
# option A (2026-11-26): no shuffled ORC job; the closed cells compute the directed ORC in closed form, and
# C5c computes the shuffled stream's coupling itself
release("C5a finished")

In [ ]:
# C5b (GPU; set CITECURV_GPU_STEP=1 before cell 1): score the shuffled stream with seed 0's checkpoint
import torch

from citecurv.tgn.infer import infer_tgn

MAX_HOURS = 2
watchdog = start_watchdog(MAX_HOURS)
gpu = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0)}
print(gpu)
SAME_STACK = gpu["python"] == "3.13.16" and gpu["torch"] == "2.14.1+cu130" and "L4" in gpu["gpu"]
if not SAME_STACK:
    # fallback: re-score the ORIGINAL dataset here and measure how far this stack is from the stored scores
    import numpy as np

    from citecurv.tgn.infer import load_event_scores

    chk = DRIVE / "tgn" / "infer" / "TGN_full_seed0_restack"
    again, _ = infer_tgn(DRIVE / "tgn" / "runs" / "TGN_full_seed0", DRIVE / "tgn" / "data", "full", out_dir=chk,
                         device="cuda", seed=0, embeddings_dtype="float32", save_embeddings=False)
    stored = load_event_scores(DRIVE / "tgn" / "infer" / "TGN_full_seed0").set_index("event_id")
    a2 = again.set_index("event_id").loc[stored.index]
    cross = float(max(np.abs(a2["p_link"] - stored["p_link"]).max(), np.abs(a2["neg_p_link"] - stored["neg_p_link"]).max()))
    print("cross-stack maximum", cross)
drift_tol = 1e-12 if SAME_STACK else 10 * cross
OUT = DRIVE / "tgn" / "infer" / "TGN_full_seed0_shuffled777"
scores, _ = infer_tgn(DRIVE / "tgn" / "runs" / "TGN_full_seed0", DRIVE / "tgn" / "data_shuf777", "full_shuf777",
                      out_dir=OUT, device="cuda", seed=0, embeddings_dtype="float32")
(OUT / "gpu_stack.json").write_text(json.dumps({**gpu, "same_stack": SAME_STACK, "drift_tol": drift_tol}))
release("C5b finished")

In [ ]:
# C5c (CPU, one session): base run, merge, shuffled run, gate
from citecurv.eval import report_s43 as R
from citecurv.eval.ablation import write_run
from citecurv.curvature.overlap import reference_overlap
from citecurv.pipeline import gate_shuffle_tgn, run_pipeline, shuffle_test_timestamps
from citecurv.tgn.infer import merge_shuffled_tgn

DRIFT_TOL = json.loads((DRIVE / "tgn" / "infer" / "TGN_full_seed0_shuffled777" / "gpu_stack.json").read_text())["drift_tol"]
MAX_HOURS = 12
watchdog = start_watchdog(MAX_HOURS)
load_inputs()
# order (memory): both couplings first, then the base run, then drop the base coupling, then the shuffled arrays
base_coupling = coupling()
shuf_ev = shuffle_test_timestamps(INPUTS["events"], CUTOFF, seed=777)
MEM.set("coupling_shuffled")
shuf_coupling = reference_overlap(_canonical_events(shuf_ev), context=INPUTS["context"])
print("couplings ready; VmRSS", round(MemLog.rss_gb(), 2), "GB", flush=True)
orig = tgn_inputs(0)
base = run_pipeline(**INPUTS["data"], **orig, variant="full", seed=0, cfg=INPUTS["cfg"],
                    context=INPUTS["context"], coupling=base_coupling, tags=tags_for(0))
INPUTS.pop("coupling", None)
del base_coupling
gc.collect()
print("base run done; VmRSS", round(MemLog.rss_gb(), 2), "GB", flush=True)
shuf_dir = DRIVE / "tgn" / "infer" / "TGN_full_seed0_shuffled777"
with np.load(shuf_dir / "tgn_event_embeddings.npz") as z:
    shuf_emb = {k: z[k] for k in z.files}
m_scores, m_emb, merge_report = merge_shuffled_tgn(orig["tgn_scores"], orig["tgn_embeddings"],
                                                   load_event_scores(shuf_dir), shuf_emb, INPUTS["events"], CUTOFF)
del shuf_emb, orig
gc.collect()
shuffled = run_pipeline(**{**INPUTS["data"], "events": shuf_ev}, tgn_scores=m_scores, tgn_embeddings=m_emb,
                        variant="full", seed=0, cfg=INPUTS["cfg"], context=INPUTS["context"],
                        coupling=shuf_coupling, tags=tags_for(0, {"group": "matrix"}))
timestamp_shuffle = gate_shuffle_tgn(base, shuffled, merge_report, drift_tol=DRIFT_TOL, event_labels=INPUTS["labels"])
write_run(MATRIX_ROOT / "runs" / "shuffle_full_0", dict(shuffled.metrics, kind="sanity",
                                                                     method="shuffle_full"))
write_run(MATRIX_ROOT / "runs" / "gates_shuffle_0",
          {"kind": "sanity", "method": "gates", "seed": 0, "tags": {"group": "full_run_gates", "level": "real"},
           "test": None, "gates": {"timestamp_shuffle": timestamp_shuffle}, "merge_report": merge_report})
print(timestamp_shuffle["status"], {k: timestamp_shuffle["value"].get(k) for k in
                                    ("auroc_full", "auroc_shuffled", "drift_ok", "pre_max_abs_diff_p_link")})
ev_now = _canonical_events(INPUTS["events"])
keys = R.cluster_keys(ev_now, INPUTS["labels"])
iv = R.paired_paper_bootstrap([(base.scores, shuffled.scores)], INPUTS["labels"], keys, metric="auroc", n_draws=1000)
write_run(MATRIX_ROOT / "runs" / "report_shuffle_components_0",
          R.shuffle_components_record(timestamp_shuffle["value"], iv, 0, []))
print("AUROC change base - shuffled:", iv["delta"], "interval", iv["lo"], iv["hi"])
if timestamp_shuffle["status"] != "PASS":
    print("STOP: report to Maor; no tables until he rules")
release("C5c finished")

## C6 - derived records and the tables

In [ ]:
from citecurv.data.slice_query import mass_events, prepare_records
from citecurv.eval import report_s43 as R

MAX_HOURS = 12
watchdog = start_watchdog(MAX_HOURS)
load_inputs()
ev = _canonical_events(INPUTS["events"])
split = split_events(ev, CUTOFF, val_frac=INPUTS["rcfg"]["split"]["val_frac"]).to_numpy(dtype=object)
work = EVAL / "report_inputs_orcdir"
work.mkdir(exist_ok=True)
# G12: citations of ids without an OpenAlex record (cited-only ids missing from cited_joint)
found = set(pd.read_parquet(DRIVE / "cited_joint.parquet", columns=["work_id"])["work_id"].astype(str))
cited_only = set(ev["dst_id"].astype(str)) - set(INPUTS["works"]["work_id"].astype(str))
R.mask_without_recordless(ev, cited_only - found).rename("keep").reset_index().rename(
    columns={"index": "event_id"}).to_parquet(work / "mask_without_recordless.parquet")
keep, src_seen = R.mask_unseen_cited(ev, split)
keep.rename("keep").reset_index().rename(columns={"index": "event_id"}).to_parquet(work / "mask_unseen_cited.parquet")
print("unseen_cited: share of test events whose citing work appears in training:", src_seen)
NOTES = {"unseen_cited": f"share of test events whose citing work appears in training: {src_seen:.4f}"}
# G9 batches: BATCH_PAPERS = {level: [work ids]} and PAPER_BATCH = {work_id: batch id}, prepared from the RWDB
# snapshot with mass_events (min_size 50) and labeled_works_openalex.parquet (SET: path of the small file built on the
# laptop from the local RWDB snapshot, uploaded on Maor's word)
BATCH_FILE = DRIVE / "batches_s43.json"
batch_info = json.loads(BATCH_FILE.read_text())
for level, papers in batch_info["levels"].items():
    m_ = R.mask_without_batches(ev, INPUTS["labels"], set(papers))
    m_.rename("keep").reset_index().rename(columns={"index": "event_id"}).to_parquet(
        work / f"mask_without_top_batches_{level}.parquet")
    NOTES[f"without_top_batches_{level}"] = (f"batches {batch_info.get('batches', {}).get(level)}; "
                                             f"events dropped {int((~m_).sum())}")
(work / "paper_batch.json").write_text(json.dumps(batch_info["paper_batch"]))
# G7 controls (higher = more suspicious before the validation sign rule)
ov = pd.read_parquet(DRIVE / "overlap_cache.parquet")
pd.DataFrame({"event_id": ov["event_id"], "score": -ov["overlap"]}).to_parquet(work / "ctl_coupling_share.parquet")
from citecurv.curvature.screen import forman_screen

scr = forman_screen(ev, unknown_without_refs=False, context=INPUTS["context"], return_degrees=True)
# option A: the exact-only view, rows whose citing work has no citer at t (Proposition 2 holds there exactly)
pd.DataFrame({"event_id": scr["event_id"], "keep": scr["in_deg_src"].to_numpy() == 0}).to_parquet(
    work / "mask_citing_uncited.parquet")
NOTES["citing_uncited"] = ("rows where the citing work has no citer at t; orc_dir is the closed form elsewhere "
                           f"too (Maor 2026-11-26), kept {int((scr['in_deg_src'] == 0).sum())} of {len(scr)}")
# the cited work's reference count at t (out-degree of dst), NaN where it has no known reference
pd.DataFrame({"event_id": scr["event_id"], "score": np.where(scr["ref_known"], scr["out_deg_dst"], np.nan)}).to_parquet(
    work / "ctl_cited_ref_count.parquet")
n_refs = ev.groupby("src_id")["dst_id"].transform("size")
pd.DataFrame({"event_id": ev["event_id"], "score": n_refs.to_numpy(dtype=float)}).to_parquet(
    work / "ctl_citing_ref_count.parquet")
first_seen = pd.concat([ev[["src_id", "t"]].rename(columns={"src_id": "w"}),
                        INPUTS["context"][["src_id", "t"]].rename(columns={"src_id": "w"})]).groupby("w")["t"].min()
age = ev["t"].to_numpy() - ev["dst_id"].map(first_seen).to_numpy(dtype=float)
pd.DataFrame({"event_id": ev["event_id"], "score": -age}).to_parquet(work / "ctl_dst_age.parquet")
pd.DataFrame({"event_id": scr["event_id"], "score": scr["ref_known"].astype(float)}).to_parquet(
    work / "ctl_ref_known.parquet")
pd.DataFrame({"event_id": scr["event_id"], "ref_known": scr["ref_known"]}).to_parquet(work / "known.parquet")
ev.to_parquet(work / "events.parquet")
free_inputs()  # the report runs in a child process: free this kernel's frames first
del scr, ov, first_seen, age, n_refs
gc.collect()
spec = {"runs": str(MATRIX_ROOT), "labels": str(DRIVE / "event_labels.parquet"),
        "events": str(work / "events.parquet"), "seeds": SEEDS, "n_draws": 1000, "mask_notes": NOTES,
        "aliases": {"no_temporal_keep_temp": [str(KEEP_ROOT), "no_temporal"]},
        "batches": str(work / "paper_batch.json"), "known": str(work / "known.parquet"),
        "masks": {p.stem.replace("mask_", ""): str(p) for p in work.glob("mask_*.parquet")},
        "controls": {p.stem.replace("ctl_", ""): str(p) for p in work.glob("ctl_*.parquet")}}
(work / "spec.json").write_text(json.dumps(spec, indent=1))
!cd {REPO_DIR} && python -m citecurv.eval.report_s43 {work}/spec.json
BANNER = ("PROVISIONAL settings: s_geom = closed-form directed ORC (option A, 2026-11-26; exact where the citing "
          "work has no citer at t, view citing_uncited), cold-start k = 3; G11 "
          "(total_cold departs from L19, advisors not yet told); G3 (label rule not yet signed off)")
!cd {REPO_DIR} && python -m citecurv.eval.ablation --tables {MATRIX_ROOT} --out {DRIVE}/eval_results_orcdir --banner "{BANNER}"
release("C6 finished")